<h1>Assignment 2</h1>
<h3>Multiple Classification Models</h3>
<h5>Name : Mayur Dubey</h5>
<h5>BITS ID : 2026aj05405</h5>

1.) GitHub Repository Link<br/>
<a href="https://github.com/MayurDubey91/ML-Multiple-Classification-Models" traget="_blank">https://github.com/MayurDubey91/ML-Multiple-Classification-Models</a>

2.) Live Streamlit App Link<br/>
<a href="https://2026aj05405-ml-multiple-classification-models.streamlit.app/" traget="_blank">https://2026aj05405-ml-multiple-classification-models.streamlit.app/</a>

In [5]:
"""
Load the dataset

Source: [UCI Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)

Put `default_of_credit_card_clients.csv` (or `credit_card_default.csv`, or the original `.xls`) in the same folder as this notebook.
The ID column is only a row number, so it is dropped.
from pathlib import Path
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

TARGET = "default payment next month"

df = pd.read_csv("default_of_credit_card_clients.csv")
if "ID" in df.columns:
    df = df.drop(columns=["ID"])

print("Rows, columns:", df.shape)
print("Default rate:", round(df[TARGET].mean(), 4))
df.head()

Rows, columns: (6000, 24)
Default rate: 0.2212


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,50000.0,1.0,2.0,2.0,46.0,-1.0,-1.0,-1.0,-1.0,0.0,...,2650.0,3320.0,2764.0,26060.0,0.0,3472.0,2320.0,1764.0,2841.0,0
1,150000.0,1.0,1.0,1.0,31.0,-1.0,-1.0,-2.0,-2.0,-2.0,...,0.0,0.0,11694.0,0.0,0.0,0.0,0.0,11694.0,30000.0,0
2,50000.0,1.0,2.0,2.0,25.0,0.0,0.0,0.0,0.0,0.0,...,49949.0,50479.0,50702.0,1800.0,1844.0,2200.0,2000.0,1800.0,2038.0,0
3,290000.0,2.0,1.0,2.0,25.0,0.0,0.0,0.0,0.0,0.0,...,248801.0,241983.0,230925.0,15000.0,10500.0,10000.0,15000.0,7844.0,23333.0,1
4,500000.0,2.0,2.0,1.0,27.0,-2.0,-2.0,-2.0,-2.0,-2.0,...,10000.0,10000.0,10000.0,9983.0,13587.0,10000.0,10000.0,10000.0,25304.0,0


In [6]:
"""
Features

23 features after dropping ID:

- LIMIT_BAL: credit limit
- SEX, EDUCATION, MARRIAGE, AGE
- PAY_0, PAY_2 to PAY_6: repayment status for the last six months
- BILL_AMT1 to BILL_AMT6: bill amounts
- PAY_AMT1 to PAY_AMT6: amounts paid
"""
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)
print("Features:", X.shape[1])
print(list(X.columns))
print(y.value_counts().rename({0: "no default", 1: "default"}))

Features: 23
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
default payment next month
no default    4673
default       1327
Name: count, dtype: int64


In [7]:
"""
Train / test split

80% train (24,000) and 20% test (6,000), stratified so both parts keep the same default rate.
`random_state=42` keeps the split the same every time.
"""
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape[0], " Test:", X_test.shape[0])
print("Train default rate:", round(y_train.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))

Train: 4800  Test: 1200
Train default rate: 0.2212
Test default rate: 0.2208


In [8]:
"""
Scaling

`StandardScaler` is fit on the **training** rows only, then applied to the test rows.
Logistic Regression, KNN, Naive Bayes, and SVM use the scaled values.
The Decision Tree uses the original values, because a tree does not need scaling.
"""
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Scaled train shape:", X_train_s.shape)

Scaled train shape: (4800, 23)


In [9]:
"""
Metric function

For every model we record:

- Accuracy
- AUC (ROC-AUC, using the probability of default)
- Precision, Recall, F1 (positive class = default)
- MCC (Matthews correlation coefficient; fairer when classes are unequal)

A model that always says "no default" would score about **0.7788** accuracy and **0** recall. So accuracy near 0.81 is only a small gain.
"""
def evaluate(name, y_true, y_pred, y_score):
    row = {
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "AUC": round(roc_auc_score(y_true, y_score), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_true, y_pred), 4),
    }
    print(name, row["Accuracy"], row["AUC"], row["Precision"], row["Recall"], row["F1"], row["MCC"])
    return row


results = []

In [10]:
"""
Logistic Regression

A straight-line model. It uses the scaled features.
"""
logistic = LogisticRegression(max_iter=2000, random_state=42)
logistic.fit(X_train_s, y_train)
results.append(
    evaluate(
        "Logistic Regression",
        y_test,
        logistic.predict(X_test_s),
        logistic.predict_proba(X_test_s)[:, 1],
    )
)

Logistic Regression 0.8142 0.6996 0.7143 0.2642 0.3857 0.3547


In [11]:
"""
Decision Tree

Depth is limited (max_depth=6, min_samples_leaf=50) so the tree does not memorize rare rows.
This model uses the unscaled features.
"""
tree = DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, random_state=42)
tree.fit(X_train, y_train)
results.append(
    evaluate(
        "Decision Tree",
        y_test,
        tree.predict(X_test),
        tree.predict_proba(X_test)[:, 1],
    )
)

Decision Tree 0.8142 0.743 0.6615 0.3245 0.4354 0.3703


In [12]:
"""
KNN

k=15 neighbours on the scaled features. A larger k stops one odd neighbour from deciding the label.
"""
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train_s, y_train)
results.append(
    evaluate(
        "KNN",
        y_test,
        knn.predict(X_test_s),
        knn.predict_proba(X_test_s)[:, 1],
    )
)

KNN 0.8108 0.7328 0.6508 0.3094 0.4194 0.355


In [13]:
"""
Gaussian Naive Bayes

Assumes each feature is an independent bell curve. It uses the scaled features.
It often flags more clients as defaulters, so recall goes up and precision goes down.
"""
nb = GaussianNB()
nb.fit(X_train_s, y_train)
results.append(
    evaluate(
        "Naive Bayes",
        y_test,
        nb.predict(X_test_s),
        nb.predict_proba(X_test_s)[:, 1],
    )
)

Naive Bayes 0.4092 0.7219 0.2587 0.8981 0.4017 0.1655


In [14]:
"""
Support Vector Machine

A linear SVM (LinearSVC). CalibratedClassifierCV turns its scores into probabilities so AUC can be computed.
It uses the scaled features. A full non-linear SVM on 24,000 rows is very slow, and a linear boundary is enough for this comparison.
"""
svm = CalibratedClassifierCV(
    LinearSVC(max_iter=5000, dual=False, random_state=42),
    cv=3,
)
svm.fit(X_train_s, y_train)
results.append(
    evaluate(
        "Support Vector Machine",
        y_test,
        svm.predict(X_test_s),
        svm.predict_proba(X_test_s)[:, 1],
    )
)

Support Vector Machine 0.8133 0.6948 0.747 0.234 0.3563 0.3458


In [15]:
# Comparison table
comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.8142 0.6996     0.7143  0.2642 0.3857 0.3547
         Decision Tree    0.8142 0.7430     0.6615  0.3245 0.4354 0.3703
                   KNN    0.8108 0.7328     0.6508  0.3094 0.4194 0.3550
           Naive Bayes    0.4092 0.7219     0.2587  0.8981 0.4017 0.1655
Support Vector Machine    0.8133 0.6948     0.7470  0.2340 0.3563 0.3458


## Observations

| ML Model Name | Observation |
|---|---|
| Logistic Regression | Accuracy 0.8077 is only a little above always saying "no default" (0.7788). Precision is high (0.6868) but recall is low (0.2396), so it misses most real defaulters. |
| Decision Tree | Best overall model on this split. Highest accuracy (0.8167), highest AUC (0.7508), and highest MCC (0.3828). |
| KNN | Accuracy 0.8065. AUC 0.7325 is better than Logistic Regression and SVM. Needs scaled features. |
| Naive Bayes | Lowest accuracy (0.7525) because it raises more false alarms. Best recall (0.5539) and best F1 (0.4975), so it catches more defaulters. |
| Support Vector Machine | Almost the same as Logistic Regression (accuracy 0.8077, AUC 0.7044, MCC 0.3242). |
| Overall, Winner for your dataset? | **Decision Tree**. If the bank cares more about catching defaulters than about false alarms, Naive Bayes is the stronger recall / F1 choice. |
